# The aleph co-training dial — one install, one API

**The question.** Snap an aleph adapter onto a small vision trunk and train
both at once. exp012 says the address-bottleneck prior *pays* when trunk and
head co-train (7/7); exp013-A says it *costs* on a frozen substrate. The
boundary between them is the dial this bed turns:

```
trainable trunk blocks:  0        1        2        4
                      frozen  ------------->  full co-training
                   (exp013 pole)           (exp012 pole)
```

**The substrate has to be able to compose.** The `linear` and `trigram`
trunks do NO cross-token mixing — every block is a per-token MLP, so the
whole model is a *generalized additive model*: two pixels outside one token
never interact, and an adapter bolted on reads a stream that was never
mixed. That is why those climbs tied `soft == none`. `input_mode="patch"` is
the architecture the bed requires: a 2D-patch ViT whose token mixer is the
**aleph router** (signed-projective addresses as a linear-attention kernel),
*not* softmax — because geometry erodes through standard attention. The
adapter now rides a mixed, non-eroding stream, exactly as RelayPatchwork
rides a real Qwen-VL / GPT-2 host.

`linear` and `trigram` remain selectable as the additive-baseline controls.

**This notebook never needs patching.** It installs the package from the
branch and calls its API. To iterate: push to `@experimental`, then re-run
the install cell with `FORCE_REINSTALL = True`.

## 1 · Install

One line. Every dependency is a *floor* that Colab already satisfies
(py3.12, torch 2.10, torchvision 0.25, numpy 2.x), so pip reinstalls
nothing binary and **no runtime restart is needed**.

In [ ]:
FORCE_REINSTALL = False   # True after pushing new commits to @experimental

import importlib, importlib.util, subprocess, sys

SPEC = ("amoe-lora[experiment,experiment-hf] @ git+https://github.com/AbstractEyes/amoe-lora@experimental")

if FORCE_REINSTALL:
    # --no-deps: refresh only our package, never touch Colab's binaries
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--force-reinstall', '--no-deps', SPEC], check=True)
    for m in [k for k in sys.modules if k.startswith(('aleph_mnist', 'amoe'))]:
        del sys.modules[m]
elif importlib.util.find_spec('aleph_mnist') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', SPEC],
                   check=True)

import aleph_mnist, amoe
print('aleph_mnist', aleph_mnist.__version__, '|', aleph_mnist.__file__)

## 2 · Probe the runtime

Print what this machine *actually* has and assert the invariants the
dependency floors rely on. If a future Colab image drifts (numpy capped
below 2, torch below 2.1, an old `huggingface_hub`), this fails loudly
**here** rather than deep inside a run.

In [ ]:
import sys, importlib, importlib.metadata as md

def _v(mod):
    try: return importlib.import_module(mod).__version__
    except Exception:
        try: return md.version(mod)
        except Exception: return 'MISSING'

print('python      ', sys.version.split()[0])
for m in ['numpy','torch','torchvision','matplotlib','datasets',
          'huggingface_hub','aleph_mnist','amoe']:
    print(f'{m:15}', _v(m))

import numpy, torch
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 1), 'torch floor'
assert not numpy.__version__.startswith('1.'), 'numpy 2.x expected on Colab'
try:
    import huggingface_hub as hh
    assert hasattr(hh, 'get_token'), 'huggingface_hub 1.x expected (HfFolder is gone)'
except ImportError:
    pass
print('\ndevice:', 'cuda' if torch.cuda.is_available() else 'cpu',
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('OK — invariants hold')

## 3 · The dial (one sweep)

`run_sweep` builds the bed, pretrains one shared phase-0 trunk per seed,
then runs `arms x dial positions`. Arms are **parameter-identical** —
`soft` (the aleph), `sign` (hard oriented code), `none` (**the control**,
a passthrough read), `off` (no adapter at all). A delta between `soft` and
`none` is attributable to the *read* and nothing else.

In [ ]:
from aleph_mnist import RunConfig, run_sweep, run_climb, smoke
from aleph_mnist.diagnostics import plots

cfg = RunConfig(
    dataset='mnist',      # mnist | fashion | cifar10
    d=64,                 # trunk width
    train_n=4096,         # None = the FULL training set
    steps=400, pretrain_steps=600,
    input_mode='patch',   # 'patch' = aleph-routed ViT (mixes; the real bed)
                          # 'trigram'/'linear' = the additive controls
    patch_size=4,         # a token is one 4x4 image region -> 49+CLS tokens
    num_heads=4,          # routed-attention heads (must divide d)
)
rows = run_sweep(cfg, seeds=(0,))
print(f'\n{len(rows)} cells')
print(plots.verdict_table(rows))

### 3b · Scratch-only (the literal co-training question)

`run_sweep(..., include_scratch=True)` also runs the **from-scratch** rows —
trunk and head moving together from step 0, no phase-0 pretrain. Those are
the co-training question in its purest form, but they need the trunk's *full*
budget: a pretrained dial cell's trunk saw `pretrain_steps + steps` of
training, so a scratch cell that runs only `steps` from random init is
undertrained by exactly `pretrain_steps` and loses on compute, not on the
address. `run_scratch` fixes that — each cell trains for `pretrain_steps +
steps` by default (pass `steps=` to override), and it skips the pretrained
dial so you can iterate on scratch alone.

In [ ]:
from aleph_mnist import run_scratch

SCRATCH_ONLY = False   # flip on to run just the from-scratch rows

if SCRATCH_ONLY:
    # fair budget by default = pretrain_steps + steps (here 600 + 400 = 1000).
    # override with steps=... if you want longer, e.g. steps=2000.
    scratch_rows = run_scratch(cfg, seeds=(0,))
    print(f'\n{len(scratch_rows)} scratch cells')
    print(plots.verdict_table(scratch_rows))
else:
    print('SCRATCH_ONLY = False — the dial above already ran the pretrained cells.')

## 4 · The substrate climb (optional, longer)

The dial walked up a width ladder across datasets — a fresh phase-0 trunk at
every width. In patch mode each image is `49–65` patch tokens and the
routed-attention blocks carry the capacity (the readout is CLS-only, so it no
longer dominates), so width `d` actually buys the model something.

CIFAR loads from the `uoft-cs/cifar10` parquet over HF's CDN, with a
torchvision fallback. Set `CLIMB = True` to run it.

In [ ]:
CLIMB = False

if CLIMB:
    climb_cfg = RunConfig(train_n=None, steps=600, pretrain_steps=600,
                          batch=256, input_mode='patch', patch_size=4,
                          num_heads=4)
    climb = run_climb(climb_cfg,
                      datasets=('mnist', 'fashion', 'cifar10'),
                      dims=(64, 128, 256), seeds=(0,))
    print(f'\nclimb: {len(climb)} cells')
    print(plots.verdict_table(climb))
else:
    print('CLIMB = False — flip it on to run the width ladder.')

## 4c · Addressed convolution — match or defeat conv2d

The patch-ViT lost to a plain conv on small vision data (it threw away conv's
locality/edge prior). `input_mode="addr_conv"` brings the geometry back
*inside* the conv: a bank of `k_bank` filters is convolved, and at every
position the **aleph read** composes a convex weight over the branches — a
content-steered, still-local, still-weight-shared convolution that reduces
**exactly** to a plain conv under a uniform address. The arms are the address
modes:

- `soft` — the aleph read · `none` — uniform (**= plain conv**, param-matched
  control) · `off` — lean plain conv · `learned` — a non-aleph dynamic conv
  (is it the aleph, or *any* dynamic conv?) · `sign` — hard branch select.

**Preregistered:** on a *classifier*, Campaign Law 2 predicts `soft ≈ none`
(the address inert — the conv sets the accuracy). The `objective="generate"`
masked-pixel phase is where the address parameterizes the output and can
genuinely pay (`soft` bpb `<` `none` bpb). The `addr KL` readout (KL of the
address to uniform) says directly whether the read is used.

In [ ]:
from aleph_mnist import run_conv

CONV = False   # flip on to run the addressed-conv head-to-head

if CONV:
    conv_cfg = RunConfig(
        dataset='mnist',            # mnist | cifar10
        input_mode='addr_conv',
        objective='classify',       # 'generate' = masked-pixel bpb (Law-2 phase)
        k_bank=16, conv_channels=32, conv_layers=2,
        train_n=4096, steps=1500, batch=128,
    )
    conv_rows = run_conv(conv_cfg, seeds=(0,),
                         arms=('soft', 'sign', 'none', 'learned', 'off'))
    print(f'\n{len(conv_rows)} conv cells')
    print(plots.verdict_table(conv_rows))
else:
    print('CONV = False — flip it on to run the addressed-conv bed.')

### 4d · Antipode conv — the address as a *signed* contribution (`conv_tokens`)

The filter-steering bed above ties because the address only re-weights a
filter-bank *mean* (a convex, hull-bounded perturbation) and it collapses the
sign. `input_mode="conv_tokens"` fixes that: a **real conv stem** produces
per-position tokens (rich k-neighbourhood features), and the **signed** aleph
read `m_hat` is **added** to each token. `m_hat` is *odd*, and the consumer is
strictly odd, so a token and its contrast-inverse push the stream in exactly
opposite directions — the antipode is the whole contribution, not a residue.

Arms isolate it: `soft` (signed antipode) vs **`mag`** (the same read on `|u|`,
sign-collapsed) — `soft − mag` attributes any effect to the **sign**; `none`
(codebook present, unread) and `off` (no read) bound the rest. The `read-amp`
readout is the contribution size (this bed ~0.1 vs the filter bed's ~0.007).

In [ ]:
from aleph_mnist import run_conv

CONV_TOKENS = False   # flip on to run the antipode-conv head-to-head

if CONV_TOKENS:
    ct_cfg = RunConfig(
        dataset='mnist',            # mnist | cifar10
        # 'conv_tokens'   = conv stem + per-token signed antipode residual
        # 'antipode_conv' = the ENTIRE conv IS the antipode read (no plain
        #                   filter, no ReLU — the read is the sole operation)
        input_mode='antipode_conv',
        objective='classify',       # 'generate' = masked-pixel bpb (Law-2 phase)
        conv_channels=64, conv_layers=2, read_layers=2, n_slots=16, k_addr=64,
        train_n=4096, steps=1500, batch=128,
    )
    ct_rows = run_conv(ct_cfg, seeds=(0,))   # arms auto = soft/mag/none/off
    print(f'\n{len(ct_rows)} cells')
    print(plots.verdict_table(ct_rows))
    for r in ct_rows:
        print(f"  {r['cell']:22} read-amp={r['addr_final'].get('read_amp_mean',0):.3f}")
else:
    print('CONV_TOKENS = False — flip it on to run the antipode-conv bed.')

## 5 · Read the result

The headline is `Δ(soft − none)`: where — if anywhere — it crosses zero is
the boundary the campaign never measured. Secondary readouts that are
worth as much as the accuracy numbers:

- **escape ratio** — domain/neutral amplitude. `≤ 1.5` means the adapter
  fires as hard on scrambled input as on real data (blend regime).
- **gate mean** vs the 0.012–0.03 live invariant band.
- **drift** toward the 0.29154 binding constant (the control's drift must
  be exactly 0 — that is how you know the arms are honest).
- **toggle damage** — what the trunk came to depend on.

In [ ]:
%matplotlib inline
fig = plots.figure_set(rows)

## 6 · Ship an anchor (the five verbs)

A result that cannot be shipped is not finished. This saves the trained
heads as a real `amoe.anchor` v1 checkpoint and puts it through the
runtime verbs on a **fresh** trunk — attach, toggle, detach-with-
verification — via the stock `binding='blocks'` path.

In [ ]:
import os, amoe
from aleph_mnist import build_model, make_bed, save_anchor, results_root
from aleph_mnist.diagnostics import probes

row = next(r for r in rows if r['cell'].startswith('soft'))
# write into results_root() so the publish cell picks it up automatically
ANCHOR = os.path.join(results_root(), 'soft.anchor.pt')
os.makedirs(results_root(), exist_ok=True)
print('anchor written, sha:', save_anchor(row, ANCHOR))

bed = make_bed(cfg)
fresh = build_model(bed, cfg)      # lands on the bed's device already
pre = probes.evaluate(fresh, bed.xte, bed.yte)   # probes align data to model
h = amoe.attach(fresh, ANCHOR, binding='blocks')        # ATTACH
with h.all_off():                                        # TOGGLE
    off = probes.evaluate(fresh, bed.xte, bed.yte)
print(f"all_off ce {off['ce']:.10f} == base {pre['ce']:.10f} ->",
      off['ce'] == pre['ce'])
h.detach(verify=True)                                    # DETACH
print('detach verified BIT-EXACT')

## Iterating

Change the **repo**, not this notebook:

1. edit + `git push` to `@experimental`
2. re-run cell 1 with `FORCE_REINSTALL = True`
3. re-run your cell

From a terminal the same runs are `aleph-mnist --smoke`,
`aleph-mnist --patch --dataset mnist`, `aleph-mnist --big --patch`, and
`aleph-mnist --patch --publish` to push the evidence to HuggingFace.
Ledgers land in `$ALEPH_RESULTS` (default `./results`).

In [ ]:
from aleph_mnist import publish

# 1) dry run — no token, no network: exactly what WOULD upload
plan = publish(dry_run=True)
print('would upload to', plan['repo_id'], '->', plan['run_path'])
for f in plan['files']:
    print('   ', f)

# 2) the real push (needs HF_TOKEN in Colab secrets). Uncomment to run:
# res = publish(repo_id='AbstractPhil/geolip-amoe-classification',
#               private=True)      # private until you choose to share it
# print('published:', res['url'])

## Iterating

Change the **repo**, not this notebook:

1. edit + `git push` to `@experimental`
2. re-run cell 1 with `FORCE_REINSTALL = True`
3. re-run your cell

From a terminal the same runs are `aleph-mnist --smoke`,
`aleph-mnist --dataset mnist --seeds 0 1`, `aleph-mnist --big --trigram`.
Ledgers land in `$ALEPH_RESULTS` (default `./results`).